# Validate Deployed Sagemaker Endpoints

## Imports

In [2]:
import json
import base64
import boto3
from pathlib import Path
from collections import Counter

## Configuration

In [3]:
# SageMaker endpoint name — adjust to match your deployed endpoint
ENDPOINT_NAME = "small-animal-classifier-concurrency-20"
REGION = "us-west-2"

session = boto3.Session(profile_name="animl")
runtime = session.client("sagemaker-runtime", region_name=REGION)

## Prepare Local Images for Inference

In [8]:
image_dir = Path("./validation/images")
images_b64 = {}
for image_path in image_dir.glob("*.jpg"):
    with open(image_path, "rb") as image_bytes:
        image_bytes = base64.b64encode(image_bytes.read()).decode("utf-8")
        images_b64[image_path.name] = image_bytes

print(f"loaded and encoded {len(images_b64)} images from {image_dir.resolve()}")

loaded and encoded 6 images from /Users/jesseleung/Projects/tnc-projects/animl/animl-ml/models/small-animal-classifier/validation/images


## Get Inference Results from Sagemaker

In [13]:
inference_res = {}
for image_name, image_b64 in images_b64.items():
    res = runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType="application/json",
        Body=json.dumps({"image": image_b64})
    )
    res = json.loads(res["Body"].read())
    inference_res[image_name] = res

## Compare to Sample Output

In [15]:
with open("./validation/sample_output.json") as f:
    sample_data = json.load(f)
sample_inference = sample_data["images"]
class_names = sample_data["classification_categories"]

for image_name, res in inference_res.items():
    sample_res = next(sample for sample in sample_inference if sample["file"] == image_name)
    if sample_res is None:
        raise Exception("Did not compare the same files")
        
    sample_classifications = sample_res["detections"][0]["classifications"]
    top_3 = sorted(res.items(), key=lambda x: x[1], reverse=True)[:3]

    for classification, confidence in top_3:
        match = next(c for c in sample_classifications if class_names[c[0]] == classification)
        if match is None:
            raise Exception("Top 3 classifications did not match")
        if abs(match[1] - confidence) >= 0.00005:
            print(match[1], confidence)
            raise Exception(f"Top 3 confidence scores did not match. Expected: {match[1]}, got: {confidence}")
        print(f"Match! Expected: {match[1]}, got: {round(confidence, 4)}, rounded from: {confidence}") 

print("Classifications and confidence scores matched sample data")

Match! Expected: 0.542, got: 0.542, rounded from: 0.5419602990150452
Match! Expected: 0.0665, got: 0.0665, rounded from: 0.0665227621793747
Match! Expected: 0.038, got: 0.038, rounded from: 0.037992607802152634
Match! Expected: 0.8005, got: 0.8005, rounded from: 0.8005000352859497
Match! Expected: 0.0282, got: 0.0282, rounded from: 0.028188753873109818
Match! Expected: 0.0151, got: 0.0151, rounded from: 0.015063728205859661
Match! Expected: 0.8556, got: 0.8556, rounded from: 0.8556024432182312
Match! Expected: 0.0203, got: 0.0203, rounded from: 0.020347336307168007
Match! Expected: 0.0172, got: 0.0172, rounded from: 0.01724955067038536
Match! Expected: 0.9008, got: 0.9008, rounded from: 0.9007808566093445
Match! Expected: 0.0124, got: 0.0124, rounded from: 0.012401514686644077
Match! Expected: 0.0109, got: 0.0109, rounded from: 0.010881843976676464
Match! Expected: 0.7737, got: 0.7737, rounded from: 0.7737169861793518
Match! Expected: 0.033, got: 0.033, rounded from: 0.0329726561903953

## Run Against Random Sample from California Small Animals

In [16]:
downloaded_image_dir = Path("./validation/images/downloaded")
downloaded_images_b64 = {}
for image_path in downloaded_image_dir.glob("*.jpg"):
    with open(image_path, "rb") as image_bytes:
        image_bytes = base64.b64encode(image_bytes.read()).decode("utf-8")
        downloaded_images_b64[image_path.name] = image_bytes

print(f"loaded and encoded {len(downloaded_images_b64)} images from {downloaded_image_dir.resolve()}")

loaded and encoded 100 images from /Users/jesseleung/Projects/tnc-projects/animl/animl-ml/models/small-animal-classifier/validation/images/downloaded


## Get Inference for Sample

In [17]:
downloaded_inference_res = {}
completed = 0
for image_name, image_b64 in downloaded_images_b64.items():
    res = runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType="application/json",
        Body=json.dumps({"image": image_b64})
    )
    res = json.loads(res["Body"].read())
    downloaded_inference_res[image_name] = res
    completed += 1
    if completed % 10 == 0:
        print(f"completed {completed} images")

completed 10 images
completed 20 images
completed 30 images
completed 40 images
completed 50 images
completed 60 images
completed 70 images
completed 80 images
completed 90 images
completed 100 images


In [22]:
with open("./validation/downloaded_output.json") as f:
    downloaded_sample_data = json.load(f)
downloaded_sample_inference = downloaded_sample_data["images"]
downloaded_class_names = downloaded_sample_data["classification_categories"]

unmatched = []
for image_name, res in downloaded_inference_res.items():
    sample_res = next(sample for sample in downloaded_sample_inference if sample["file"] == image_name)
    if sample_res is None:
        raise Exception("Did not compare the same files")
        
    sample_classifications = sample_res["detections"][0]["classifications"]
    top_3 = sorted(res.items(), key=lambda x: x[1], reverse=True)[:3]

    for classification, confidence in top_3:
        match = next(c for c in sample_classifications if downloaded_class_names[c[0]] == classification)
        if match is None:
            raise Exception("Top 3 classifications did not match")
        if abs(match[1] - confidence) >= 0.00005:
            print(f"Top 3 confidence scores did not match. Expected: {match[1]}, got: {confidence}")
            unmatched.append({"image": image_name, "expected": match[1], "got": confidence})
        else: 
            print(f"Match! Expected: {match[1]}, got: {round(confidence, 4)}, rounded from: {confidence}") 

print(f"Total matches: {len(downloaded_inference_res) - len(unmatched)}")
print(f"Total mismatches: {len(unmatched)}")
for un in unmatched:
    print(f"Image: {un["image"]}, expected: {un["expected"]}, got: {un["got"]}")

Match! Expected: 0.6141, got: 0.6141, rounded from: 0.6140721440315247
Match! Expected: 0.0549, got: 0.0549, rounded from: 0.05494869127869606
Match! Expected: 0.0316, got: 0.0316, rounded from: 0.0316198468208313
Match! Expected: 0.8977, got: 0.8977, rounded from: 0.8976628184318542
Match! Expected: 0.016, got: 0.016, rounded from: 0.016048189252614975
Match! Expected: 0.0082, got: 0.0082, rounded from: 0.00820425059646368
Top 3 confidence scores did not match. Expected: 0.4796, got: 0.4795496165752411
Match! Expected: 0.1265, got: 0.1265, rounded from: 0.12650302052497864
Match! Expected: 0.0577, got: 0.0577, rounded from: 0.05772412195801735
Match! Expected: 0.6358, got: 0.6358, rounded from: 0.6358238458633423
Match! Expected: 0.0511, got: 0.0511, rounded from: 0.05110527202486992
Match! Expected: 0.0296, got: 0.0296, rounded from: 0.029596010223031044
Match! Expected: 0.6162, got: 0.6162, rounded from: 0.6161514520645142
Match! Expected: 0.0554, got: 0.0554, rounded from: 0.055413